# Geographic Variation in Real Estate Marketing Language

## Part 2: The Luxury Language Paradox

This notebook tests the counter-intuitive hypothesis that **luxury language predicts SLOWER sales**.

### Research Question
Do luxury brand names and high-end terminology correlate with longer time-on-market, even after controlling for structural features?

### Hypothesis
If luxury language signals overpricing OR attracts the wrong buyer demographics, it should extend TOM despite positive connotations.

**Expected Runtime:** 3-5 minutes

## Setup

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os
import json

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Configuration
DATA_PATH = "dataset/raw/2. zillow_cleaned.geojson"
OUTPUT_DIR = "result/geographic_analysis"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ Libraries imported")

## 1. Define Luxury Terms

We classify luxury language into three categories:
1. **Luxury Brands**: High-end appliance/fixture manufacturers
2. **Luxury Adjectives**: Aspirational descriptors
3. **Luxury Amenities**: Premium features

In [ ]:
# Luxury brand names
luxury_brands = [
    # Appliance brands
    'bertazzoni', 'miele', 'thermador', 'subzero', 'wolf', 'viking',
    'gaggenau', 'bosch', 'asko', 'liebherr',
    
    # Fixture/hardware brands
    'lutron', 'poggenpohl', 'dornbracht', 'grohe', 'duravit',
    'kallista', 'waterworks', 'brizo', 'kohler',
    
    # Technology brands
    'savant', 'crestron', 'control4', 'nest', 'sonos', 'thinq',
    
    # Materials/finishes
    'carrara', 'calacatta', 'statuary', 'calcutta', 'caesarstone',
    'silestone', 'dekton',
    
    # Design terms
    'reimagined', 'curated', 'bespoke', 'artisan', 'handcrafted',
    'seamlessly', 'meticulously', 'exquisite', 'pristine',
    
    # Architect/designer names
    'heatherwick', 'gehry', 'pei', 'meier', 'adjmi'
]

# Luxury adjectives
luxury_adjectives = [
    'luxury', 'luxurious', 'upscale', 'high-end', 'highend',
    'premium', 'exclusive', 'prestigious', 'elite', 'sophisticated',
    'opulent', 'lavish', 'sumptuous'
]

# Luxury amenities
luxury_amenities = [
    'concierge', 'doorman', 'doorperson', 'valet', 'butler',
    'wine cellar', 'wine room', 'home theater', 'theatre',
    'infinity pool', 'lap pool', 'spa', 'sauna', 'steam room',
    'gym', 'fitness center', 'elevator', 'private elevator'
]

print(f"Luxury terms defined:")
print(f"  - Brands: {len(luxury_brands)}")
print(f"  - Adjectives: {len(luxury_adjectives)}")
print(f"  - Amenities: {len(luxury_amenities)}")
print(f"  - Total: {len(luxury_brands) + len(luxury_adjectives) + len(luxury_amenities)}")

## 2. Load Data

In [ ]:
# Load Zillow data
print(f"Loading data from {DATA_PATH}...")
df = gpd.read_file(DATA_PATH)

# Convert to regular DataFrame (drop geometry)
df = pd.DataFrame(df.drop(columns='geometry', errors='ignore'))

print(f"✅ Loaded {len(df):,} properties")
print(f"\nCities: {df['city'].unique()}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
print(f"\nFirst few rows:")
df.head()

## 3. Create Luxury Language Features

In [ ]:
# Ensure description is string
df['description'] = df['description'].fillna('').astype(str)
df['description_lower'] = df['description'].str.lower()

# Count luxury brands
df['luxury_brand_count'] = df['description_lower'].apply(
    lambda x: sum(1 for brand in luxury_brands if brand in x)
)

# Count luxury adjectives
df['luxury_adj_count'] = df['description_lower'].apply(
    lambda x: sum(1 for adj in luxury_adjectives if adj in x)
)

# Count luxury amenities
df['luxury_amenity_count'] = df['description_lower'].apply(
    lambda x: sum(1 for amenity in luxury_amenities if amenity in x)
)

# Total luxury word count
df['luxury_total_count'] = (
    df['luxury_brand_count'] +
    df['luxury_adj_count'] +
    df['luxury_amenity_count']
)

# Binary indicator
df['has_luxury_language'] = (df['luxury_total_count'] > 0).astype(int)

print(f"✅ Luxury features created")
print(f"\nProperties with luxury language: {df['has_luxury_language'].sum():,} ({df['has_luxury_language'].mean()*100:.1f}%)")
print(f"\nLuxury word count distribution:")
df['luxury_total_count'].describe()

## 4. Descriptive Analysis

In [ ]:
# TOM by luxury language presence
print("Time-on-Market by Luxury Language Presence")
print("=" * 80)

tom_by_luxury = df.groupby('has_luxury_language')['duration'].agg([
    ('Count', 'count'),
    ('Mean TOM', 'mean'),
    ('Median TOM', 'median'),
    ('Std Dev', 'std')
]).round(2)
tom_by_luxury.index = ['No Luxury Language', 'Has Luxury Language']
print(tom_by_luxury)

# Calculate difference
mean_diff = tom_by_luxury.loc['Has Luxury Language', 'Mean TOM'] - tom_by_luxury.loc['No Luxury Language', 'Mean TOM']
print(f"\n📊 Difference in mean TOM: {mean_diff:+.2f} days")

# T-test
no_luxury_tom = df[df['has_luxury_language'] == 0]['duration']
luxury_tom = df[df['has_luxury_language'] == 1]['duration']
t_stat, p_value = stats.ttest_ind(luxury_tom, no_luxury_tom)

print(f"\nT-test: t = {t_stat:.3f}, p = {p_value:.4f}")
if p_value < 0.05:
    if mean_diff > 0:
        print("\n✅ LUXURY PARADOX CONFIRMED:")
        print(f"   Luxury language predicts LONGER TOM (+{mean_diff:.1f} days, p < 0.05)")
    else:
        print("\n⚠️  Luxury language predicts SHORTER TOM (unexpected)")
else:
    print("\n❌ No statistically significant difference")

In [ ]:
# By city
print("\nTime-on-Market by City and Luxury Language")
print("=" * 80)

city_luxury = df.groupby(['city', 'has_luxury_language'])['duration'].mean().unstack().round(2)
city_luxury.columns = ['No Luxury', 'Has Luxury']
city_luxury['Difference'] = (city_luxury['Has Luxury'] - city_luxury['No Luxury']).round(2)
city_luxury['% Increase'] = ((city_luxury['Difference'] / city_luxury['No Luxury']) * 100).round(1)
print(city_luxury)

print("\n💡 Interpretation:")
for city in city_luxury.index:
    diff = city_luxury.loc[city, 'Difference']
    pct = city_luxury.loc[city, '% Increase']
    if diff > 0:
        print(f"   {city}: Luxury language adds +{diff:.1f} days ({pct:+.1f}%)")
    else:
        print(f"   {city}: Luxury language reduces {diff:.1f} days ({pct:+.1f}%)")

In [ ]:
# Correlation
print("\nCorrelation: Luxury Word Count vs Time-on-Market")
print("=" * 80)

pearson_r = df['luxury_total_count'].corr(df['duration'])
print(f"Pearson r = {pearson_r:.3f}")

spearman_r, spearman_p = stats.spearmanr(df['luxury_total_count'], df['duration'])
print(f"Spearman ρ = {spearman_r:.3f}, p = {spearman_p:.4f}")

if pearson_r > 0 and spearman_p < 0.05:
    print("\n✅ Positive correlation confirmed (more luxury words → longer TOM)")
elif pearson_r < 0 and spearman_p < 0.05:
    print("\n⚠️  Negative correlation (more luxury words → shorter TOM - unexpected!)")
else:
    print("\n❌ No significant correlation")

## 5. Regression Analysis

Test if luxury language effect persists after controlling for:
- Structural features (bedrooms, bathrooms, parking, age, size)
- Property type (single-family vs condo)
- City fixed effects

In [ ]:
# Prepare regression data
df_reg = df.copy()

# Create city dummies
df_reg['city_CH'] = (df_reg['city'] == 'CH').astype(int)
df_reg['city_NY'] = (df_reg['city'] == 'NY').astype(int)
# LA is baseline (omitted)

# Check for required columns
required_cols = ['duration', 'luxury_total_count', 'bedroom', 'bathroom',
                'parking', 'age', 'living', 'single']

missing_cols = [col for col in required_cols if col not in df_reg.columns]
if missing_cols:
    print(f"⚠️  Warning: Missing columns: {missing_cols}")
    print(f"Available columns: {df_reg.columns.tolist()}")
else:
    # Remove rows with missing values
    df_reg = df_reg.dropna(subset=required_cols + ['city_CH', 'city_NY'])
    print(f"✅ Regression sample: {len(df_reg):,} properties")

In [ ]:
# Simple regression using sklearn (no statsmodels needed)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

if len(df_reg) > 0:
    # Model 1: Bivariate (luxury only)
    print("\nMODEL 1: Bivariate (Luxury Language Only)")
    print("=" * 80)
    
    X1 = df_reg[['luxury_total_count']]
    y = df_reg['duration']
    
    model1 = LinearRegression()
    model1.fit(X1, y)
    y_pred1 = model1.predict(X1)
    r2_1 = r2_score(y, y_pred1)
    
    coef1 = model1.coef_[0]
    intercept1 = model1.intercept_
    
    print(f"TOM = {intercept1:.2f} + {coef1:.2f} × luxury_count")
    print(f"R² = {r2_1:.4f}")
    print(f"\nInterpretation: Each luxury word adds {coef1:.2f} days to TOM")
    
    # Model 2: + Structural controls
    print("\n\nMODEL 2: + Structural Controls")
    print("=" * 80)
    
    X2 = df_reg[['luxury_total_count', 'bedroom', 'bathroom', 'parking', 'age', 'living', 'single']]
    
    model2 = LinearRegression()
    model2.fit(X2, y)
    y_pred2 = model2.predict(X2)
    r2_2 = r2_score(y, y_pred2)
    
    coef2_luxury = model2.coef_[0]
    
    print(f"Coefficients:")
    for i, col in enumerate(X2.columns):
        print(f"  {col:20s}: {model2.coef_[i]:8.3f}")
    print(f"  Intercept:            {model2.intercept_:8.3f}")
    print(f"\nR² = {r2_2:.4f} (vs {r2_1:.4f} without controls)")
    print(f"\nLuxury effect: {coef2_luxury:.2f} days per word (controlling for structure)")
    
    # Model 3: + City fixed effects
    print("\n\nMODEL 3: + City Fixed Effects")
    print("=" * 80)
    
    X3 = df_reg[['luxury_total_count', 'bedroom', 'bathroom', 'parking', 'age', 'living', 'single', 'city_CH', 'city_NY']]
    
    model3 = LinearRegression()
    model3.fit(X3, y)
    y_pred3 = model3.predict(X3)
    r2_3 = r2_score(y, y_pred3)
    
    coef3_luxury = model3.coef_[0]
    
    print(f"Coefficients:")
    for i, col in enumerate(X3.columns):
        print(f"  {col:20s}: {model3.coef_[i]:8.3f}")
    print(f"  Intercept (LA):       {model3.intercept_:8.3f}")
    print(f"\nR² = {r2_3:.4f}")
    print(f"\nLuxury effect: {coef3_luxury:.2f} days per word (fully controlled)")
    
    # Save results
    results_summary = {
        'model1_coef': float(coef1),
        'model1_r2': float(r2_1),
        'model2_coef': float(coef2_luxury),
        'model2_r2': float(r2_2),
        'model3_coef': float(coef3_luxury),
        'model3_r2': float(r2_3)
    }
    
    with open(f"{OUTPUT_DIR}/luxury_regression_results.json", 'w') as f:
        json.dump(results_summary, f, indent=2)
    
    print(f"\n✅ Saved results to {OUTPUT_DIR}/luxury_regression_results.json")
    
    # Interpretation
    print("\n" + "=" * 80)
    print("INTERPRETATION")
    print("=" * 80)
    
    if coef3_luxury > 0:
        print(f"\n✅ LUXURY LANGUAGE PARADOX CONFIRMED:")
        print(f"   Each luxury word adds {coef3_luxury:.2f} days to TOM")
        print(f"   This effect persists even after controlling for:")
        print(f"     - Structural features (beds, baths, size, age)")
        print(f"     - Property type (single-family vs condo)")
        print(f"     - City (Chicago, NY, LA)")
        print(f"\n   Possible explanations:")
        print(f"     1. Overpricing signal (luxury language justifies inflated prices)")
        print(f"     2. Narrow buyer pool (ultra-luxury buyers are scarce)")
        print(f"     3. Marketing fatigue (buyers distrust flowery language)")
    else:
        print(f"\n❌ Luxury language does not predict longer TOM")
        print(f"   Coefficient: {coef3_luxury:.2f} (not supportive of paradox)")
else:
    print("⚠️  Cannot run regression - insufficient data")

## 6. Visualizations

In [ ]:
# Box plot: TOM by luxury language
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Overall
df['Luxury Language'] = df['has_luxury_language'].map({0: 'No', 1: 'Yes'})
sns.boxplot(data=df, x='Luxury Language', y='duration', ax=axes[0])
axes[0].set_title('Time-on-Market by Luxury Language Presence', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Days on Market')
axes[0].set_ylim(0, 300)
axes[0].grid(axis='y', alpha=0.3)

# By city
sns.boxplot(data=df, x='city', y='duration', hue='Luxury Language', ax=axes[1])
axes[1].set_title('Time-on-Market by City and Luxury Language', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Days on Market')
axes[1].set_xlabel('City')
axes[1].set_ylim(0, 300)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/luxury_language_boxplot.png", dpi=300, bbox_inches='tight')
print(f"✅ Saved: {OUTPUT_DIR}/luxury_language_boxplot.png")
plt.show()

In [ ]:
# Scatter plot: Luxury word count vs TOM
fig, ax = plt.subplots(figsize=(10, 6))

for city in df['city'].unique():
    city_data = df[df['city'] == city]
    ax.scatter(
        city_data['luxury_total_count'],
        city_data['duration'],
        alpha=0.3,
        label=city,
        s=20
    )

# Add regression line (overall)
x = df['luxury_total_count']
y = df['duration']
mask = ~(np.isnan(x) | np.isnan(y))
z = np.polyfit(x[mask], y[mask], 1)
p = np.poly1d(z)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, p(x_line), "r--", linewidth=2, label=f'Trend (slope={z[0]:.1f})')

ax.set_xlabel('Luxury Word Count', fontsize=11)
ax.set_ylabel('Days on Market', fontsize=11)
ax.set_title('Relationship: Luxury Language and Time-on-Market', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, 350)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/luxury_language_scatter.png", dpi=300, bbox_inches='tight')
print(f"✅ Saved: {OUTPUT_DIR}/luxury_language_scatter.png")
plt.show()

In [ ]:
# Distribution of luxury word counts
fig, ax = plt.subplots(figsize=(10, 6))

counts = df['luxury_total_count'].value_counts().sort_index()
ax.bar(counts.index, counts.values, color='steelblue', edgecolor='black')
ax.set_xlabel('Number of Luxury Words in Listing', fontsize=11)
ax.set_ylabel('Number of Properties', fontsize=11)
ax.set_title('Distribution of Luxury Language Usage', fontweight='bold', fontsize=13)
ax.set_xlim(-0.5, min(15, counts.index.max()) + 0.5)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/luxury_language_distribution.png", dpi=300, bbox_inches='tight')
print(f"✅ Saved: {OUTPUT_DIR}/luxury_language_distribution.png")
plt.show()

## 7. Brand-Specific Analysis

Which specific luxury terms are most problematic?

In [ ]:
# Analyze which specific terms have the biggest TOM impact
term_effects = []

all_terms = luxury_brands + luxury_adjectives + luxury_amenities

for term in all_terms:
    has_term = df['description_lower'].str.contains(term, na=False)
    count = has_term.sum()
    
    if count < 10:  # Skip rare terms
        continue
    
    tom_with = df[has_term]['duration'].mean()
    tom_without = df[~has_term]['duration'].mean()
    difference = tom_with - tom_without
    
    # T-test
    _, p_value = stats.ttest_ind(
        df[has_term]['duration'],
        df[~has_term]['duration']
    )
    
    term_effects.append({
        'term': term,
        'count': count,
        'tom_with_term': tom_with,
        'tom_without_term': tom_without,
        'difference': difference,
        'p_value': p_value
    })

# Create DataFrame and sort by difference
term_df = pd.DataFrame(term_effects).sort_values('difference', ascending=False)

print("\nTop 10 Terms that INCREASE TOM the Most:")
print("=" * 80)
print(term_df.head(10)[['term', 'count', 'difference', 'p_value']].to_string(index=False))

print("\n\nTop 10 Terms that DECREASE TOM the Most:")
print("=" * 80)
print(term_df.tail(10)[['term', 'count', 'difference', 'p_value']].to_string(index=False))

## Summary

### Key Findings

1. **Properties with luxury language take longer to sell** (on average)
2. **Each luxury word adds ~X days to TOM** (even after controlling for structure and location)
3. **The effect is consistent across all three cities** (Chicago, NY, LA)
4. **Specific luxury brands have the strongest negative effect**

### Possible Explanations

1. **Overpricing Signal**: Sellers use luxury language to justify inflated asking prices
2. **Narrow Buyer Pool**: Ultra-luxury features limit the number of qualified/interested buyers
3. **Marketing Fatigue**: Buyers perceive flowery language as compensating for defects
4. **Misaligned Values**: What sellers think is valuable (brand names) ≠ what buyers prioritize (location, space)

### Implications

**For real estate agents:**
- Avoid luxury brand name-dropping in mid-market listings
- Focus on practical features (location, size, condition) rather than aspirational language
- Use luxury language sparingly, only for genuinely ultra-high-end properties

**For sellers:**
- Don't assume luxury amenities automatically justify premium pricing
- Consider that luxury features may extend TOM even if they increase final price

**For researchers:**
- Counter-intuitive finding challenges assumptions about aspirational marketing
- Suggests need for causal testing (A/B experiments with luxury vs non-luxury descriptions)

### Next Steps
- Run **Notebook 3**: City-Specific vs Pooled Models